# Chapter 26 Walkthrough: Production ML Workflows — Pipelines, ColumnTransformer, Nested CV & Hyperparameter Tuning

**Book Reference:** Chapter 26 (Part V Capstone), Sections 26.1 – 26.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every workflow uses the bundled `mch_survey_sample` and `epi_surveillance` datasets with zero data leakage and clean temporary serialization.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & Survey/Surveillance Data
!pip install -q sharukh-tutor
from pathlib import Path
import tempfile
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    RandomizedSearchCV,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
epi_raw = tutor.load_data("epi_surveillance", as_pandas=True)
mch = mch_raw.copy()
mch["residence"] = (
    mch["residence"].astype(str).str.strip().str.title()
)
print("Loaded mch_survey_sample:", mch.shape,
      "| clusters:", mch["cluster_id"].nunique())
print("Loaded epi_surveillance:", epi_raw.shape,
      "| districts:", epi_raw["district"].nunique())


## Section 26.3.1 — Custom Scikit-Learn Transformer (`SentinelToNaNTransformer`)

In [ ]:
class SentinelToNaNTransformer(BaseEstimator, TransformerMixin):
    """Replace a survey sentinel code (e.g., 999) with np.nan."""

    def __init__(self, sentinel=999):
        self.sentinel = sentinel

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.copy().replace(self.sentinel, np.nan)
        arr = np.asarray(X, dtype=float).copy()
        arr[arr == self.sentinel] = np.nan
        return arr

raw_anc = mch_raw[["anc_visits", "maternal_hb_g_dl"]].copy()
tr = SentinelToNaNTransformer(sentinel=999)
clean_anc = tr.fit_transform(raw_anc)
print("Raw anc_visits max:", raw_anc["anc_visits"].max(),
      "| 999 count:", int((raw_anc["anc_visits"] == 999).sum()))
print("Transformed anc_visits max:", clean_anc["anc_visits"].max(),
      "| NaN count:", int(clean_anc["anc_visits"].isna().sum()))
print("Original raw_anc mutated?:",
      bool((raw_anc["anc_visits"] == 999).sum() != 8))


## Section 26.3.2 — Leak-Free `ColumnTransformer` + `Pipeline` for Mixed Data

In [ ]:
num_cols = ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]
cat_cols = ["residence", "wealth_quintile"]
X = mch[num_cols + cat_cols]
y = mch["child_stunted"]
groups = mch["cluster_id"]

num_pipe = Pipeline([
    ("sentinel", SentinelToNaNTransformer(999)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore", sparse_output=False
    )),
])
prep = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])
pipe_lr = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(max_iter=500)),
])

Z_demo = prep.fit_transform(X)
print("Prepared matrix shape:", Z_demo.shape,
      "| any NaN?:", bool(np.isnan(Z_demo).any()))
unseen_row = pd.DataFrame([{
    "maternal_age_years": 26,
    "anc_visits": 999,
    "maternal_hb_g_dl": np.nan,
    "residence": "PeriUrban",
    "wealth_quintile": "Middle",
}])
pipe_lr.fit(X, y)
prob = pipe_lr.predict_proba(unseen_row)[0, 1]
print("Unseen-category + 999 prob:", round(float(prob), 4))


## Section 26.3.3 — Matching CV Splitter to Study Design (`StratifiedKFold` vs. `GroupKFold` vs. `TimeSeriesSplit`)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_auc = cross_val_score(pipe_lr, X, y, cv=skf, scoring="roc_auc")

gkf = GroupKFold(n_splits=5)
gkf_auc = cross_val_score(
    pipe_lr, X, y, groups=groups, cv=gkf, scoring="roc_auc"
)

dhaka = (
    epi_raw[epi_raw["district"] == "Dhaka"]
    .sort_values(["year", "epi_week"])
    .reset_index(drop=True)
)
X_epi = dhaka[["rainfall_mm", "mean_temp_c", "epi_week"]]
y_epi = dhaka["dengue_cases"]
ts_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])
tscv = TimeSeriesSplit(n_splits=4)
ts_rmse = -cross_val_score(
    ts_pipe, X_epi, y_epi, cv=tscv,
    scoring="neg_root_mean_squared_error",
)
print(f"StratifiedKFold (5-fold) AUC: "
      f"{skf_auc.mean():.4f} +/- {skf_auc.std():.4f}")
print(f"GroupKFold (50 clusters) AUC: "
      f"{gkf_auc.mean():.4f} +/- {gkf_auc.std():.4f}")
print(f"TimeSeriesSplit (Dhaka) RMSE: {ts_rmse.mean():.4f} | "
      f"folds: {[round(float(r), 2) for r in ts_rmse]}")


## Section 26.3.4 — Eliminating Tuning Optimism Bias With True Nested Cross-Validation

In [ ]:
pipe_rf = Pipeline([
    ("prep", prep),
    ("model", RandomForestClassifier(
        n_estimators=25, random_state=42
    )),
])
rf_grid = {
    "model__max_depth": [2, 4, 8],
    "model__min_samples_leaf": [1, 5, 15],
}
outer_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
flat_gs = GridSearchCV(pipe_rf, rf_grid, cv=outer_cv, scoring="roc_auc")
flat_gs.fit(X, y)

nested_scores = []
for tr_i, te_i in outer_cv.split(X, y):
    inner_cv = StratifiedKFold(
        n_splits=3, shuffle=True, random_state=42
    )
    gs = GridSearchCV(
        pipe_rf, rf_grid, cv=inner_cv, scoring="roc_auc"
    )
    gs.fit(X.iloc[tr_i], y.iloc[tr_i])
    prob = gs.best_estimator_.predict_proba(X.iloc[te_i])[:, 1]
    nested_scores.append(round(float(roc_auc_score(y.iloc[te_i], prob)), 4))

bias = float(flat_gs.best_score_ - np.mean(nested_scores))
print(f"Naive Flat GridSearchCV AUC: {flat_gs.best_score_:.4f} | "
      f"best: {flat_gs.best_params_}")
print(f"True Nested CV AUC:          {np.mean(nested_scores):.4f} "
      f"+/- {np.std(nested_scores):.4f}")
print("Outer fold scores:", nested_scores)
print(f"Optimism bias (flat - nested): {bias:.4f}")


## Section 26.3.5 — Efficient Hyperparameter Search (`RandomizedSearchCV` & Bayesian/Optuna Intuition)

In [ ]:
param_dist = {
    "model__max_depth": [2, 3, 4, 6, 8, None],
    "model__min_samples_leaf": [1, 2, 5, 10, 15],
    "model__max_features": ["sqrt", "log2", 0.7],
}
rs = RandomizedSearchCV(
    pipe_rf,
    param_dist,
    n_iter=6,
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=42),
    scoring="roc_auc",
    random_state=42,
)
rs.fit(X, y)
print(f"RandomizedSearchCV (6 draws) best AUC: {rs.best_score_:.4f}")
print("Best params:", rs.best_params_)


## Section 26.3.6 — Reproducible Serialization With `joblib` Inside `tempfile.TemporaryDirectory()`

In [ ]:
X_clean = X.replace(999, np.nan)
deploy_prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", cat_pipe, cat_cols),
])
deploy_pipe = Pipeline([
    ("prep", deploy_prep),
    ("model", LogisticRegression(max_iter=500)),
])
deploy_pipe.fit(X_clean, y)
X_sample = X_clean.iloc[:5]
with tempfile.TemporaryDirectory() as tmpdir:
    artifact = Path(tmpdir) / "stunting_lr_pipeline.joblib"
    joblib.dump(deploy_pipe, artifact)
    reloaded = joblib.load(artifact)
    p_orig = deploy_pipe.predict_proba(X_sample)[:, 1]
    p_load = reloaded.predict_proba(X_sample)[:, 1]
    print("Saved artifact bytes:", artifact.stat().st_size)
    print("Predictions bit-identical?:",
          bool(np.allclose(p_orig, p_load)))
    print("Reloaded risk probs:",
          [round(float(p), 4) for p in p_load])


## Section 26.3.7 — Break It On Purpose! (Production ML Error Anatomy)

In [ ]:
# Bug #1: ValueError — unseen district with handle_unknown='error'
enc_strict = OneHotEncoder(handle_unknown="error", sparse_output=False)
enc_strict.fit(pd.DataFrame({"district": ["Dhaka", "Sylhet"]}))
enc_strict.transform(pd.DataFrame({"district": ["Khulna"]}))


In [ ]:
# Bug #2: ValueError — missing 'model__' prefix in param_grid
bad_gs = GridSearchCV(pipe_lr, {"C": [0.1, 1.0]}, cv=3)
bad_gs.fit(X, y)


In [ ]:
# Bug #3: ValueError — GroupKFold called without groups=
cross_val_score(
    pipe_lr, X, y, cv=GroupKFold(n_splits=5), scoring="roc_auc"
)


## Section 26.4 — Stage 4 Blueprint: End-to-End Cluster-Nested Audit & Artifact Packaging

In [ ]:
def audit_and_package_dhs_model(raw_df, seed=42):
    """Run Group-Nested CV, refit best DHS pipeline & verify joblib."""
    df = raw_df.copy()
    df["residence"] = (
        df["residence"].astype(str).str.strip().str.title()
    )
    n_cols = ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]
    c_cols = ["residence", "wealth_quintile"]
    df[n_cols] = df[n_cols].replace(999, np.nan)
    X_all, y_all = df[n_cols + c_cols], df["child_stunted"]
    grps = df["cluster_id"]

    n_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    c_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore", sparse_output=False
        )),
    ])
    pre = ColumnTransformer([
        ("num", n_pipe, n_cols),
        ("cat", c_pipe, c_cols),
    ])
    cand = Pipeline([
        ("prep", pre),
        ("model", LogisticRegression(max_iter=500)),
    ])

    grid = {"model__C": [0.05, 0.2, 1.0, 5.0]}
    outer = GroupKFold(n_splits=5)
    fold_aucs = []
    for tr_i, te_i in outer.split(X_all, y_all, groups=grps):
        inner = StratifiedKFold(
            n_splits=3, shuffle=True, random_state=seed
        )
        search = GridSearchCV(cand, grid, cv=inner, scoring="roc_auc")
        search.fit(X_all.iloc[tr_i], y_all.iloc[tr_i])
        pr = search.best_estimator_.predict_proba(X_all.iloc[te_i])[:, 1]
        auc_i = roc_auc_score(y_all.iloc[te_i], pr)
        fold_aucs.append(round(float(auc_i), 4))

    final_search = GridSearchCV(
        cand, grid,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=seed),
        scoring="roc_auc",
    )
    final_search.fit(X_all, y_all)
    best_pipe = final_search.best_estimator_

    with tempfile.TemporaryDirectory() as tmpdir:
        out_path = Path(tmpdir) / "dhs_production_pipe.joblib"
        joblib.dump(best_pipe, out_path)
        reloaded = joblib.load(out_path)
        same = bool(np.allclose(
            best_pipe.predict_proba(X_all.iloc[:5])[:, 1],
            reloaded.predict_proba(X_all.iloc[:5])[:, 1],
        ))
        n_bytes = int(out_path.stat().st_size)

    return {
        "nested_group_auc": round(float(np.mean(fold_aucs)), 4),
        "nested_group_std": round(float(np.std(fold_aucs)), 4),
        "best_C": final_search.best_params_["model__C"],
        "artifact_bytes": n_bytes,
        "reload_verified": same,
    }

print(audit_and_package_dhs_model(mch_raw))
